# Fine-tuning a model with the Trainer API

Install the Transformers, Datasets, and Evaluate libraries to run this notebook.

In [1]:
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorWithPadding

raw_datasets = load_dataset("glue", "mrpc")
checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)


def tokenize_function(example):
    return tokenizer(example["sentence1"], example["sentence2"], truncation=True)


tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [2]:
from transformers import TrainingArguments

training_args = TrainingArguments("test-trainer", report_to="none")

In [3]:
training_args

TrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=False,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.NO,
eval_use_gather_object=False,
f

In [4]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [5]:
from transformers import Trainer

trainer = Trainer(
    model,
    training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    processing_class=tokenizer,
)

In [6]:
trainer.train()

Step,Training Loss
500,0.603200
1000,0.481700


TrainOutput(global_step=1377, training_loss=0.4931370022509872, metrics={'train_runtime': 233.477, 'train_samples_per_second': 47.131, 'train_steps_per_second': 5.898, 'total_flos': 405114969714960.0, 'train_loss': 0.4931370022509872, 'epoch': 3.0})

In [7]:
predictions = trainer.predict(tokenized_datasets["validation"])
print(predictions.predictions.shape, predictions.label_ids.shape)

(408, 2) (408,)


In [8]:
predictions

PredictionOutput(predictions=array([[-2.165433  ,  1.9793233 ],
       [ 1.2999779 , -1.5742744 ],
       [-0.9824898 ,  0.96254706],
       [-2.1084445 ,  1.9856002 ],
       [ 1.1322777 , -1.4578364 ],
       [-2.1224866 ,  1.9939903 ],
       [-1.9005079 ,  1.7645252 ],
       [-2.1537976 ,  1.9910744 ],
       [-1.8920257 ,  1.7530615 ],
       [-2.083114  ,  1.9623249 ],
       [-2.1640408 ,  1.9802742 ],
       [ 1.1260649 , -1.4434227 ],
       [ 1.14272   , -1.3796258 ],
       [-1.9588805 ,  1.8270752 ],
       [-2.1757307 ,  1.9662751 ],
       [-2.0462637 ,  1.9209642 ],
       [-2.1685104 ,  1.9758868 ],
       [ 1.1288218 , -1.3888916 ],
       [-2.1751235 ,  1.9673883 ],
       [ 1.2150233 , -1.5332977 ],
       [ 1.2307773 , -1.5396758 ],
       [-1.4843098 ,  1.3367866 ],
       [ 0.65602034, -0.82414997],
       [-2.1448143 ,  1.995533  ],
       [-2.165376  ,  1.9783663 ],
       [-0.99590445,  1.0233678 ],
       [-1.6497687 ,  1.4690009 ],
       [-2.175397  ,  1.96

In [9]:
import numpy as np

preds = np.argmax(predictions.predictions, axis=-1)

In [10]:
import evaluate

metric = evaluate.load("glue", "mrpc")
metric.compute(predictions=preds, references=predictions.label_ids)

{'accuracy': 0.8406862745098039, 'f1': 0.8903878583473862}

In [11]:
def compute_metrics(eval_preds):
    metric = evaluate.load("glue", "mrpc")
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

In [12]:
training_args = TrainingArguments("test-trainer", report_to="none")
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

trainer = Trainer(
    model,
    training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [13]:
trainer.train()

Step,Training Loss
500,0.611200
1000,0.417600


TrainOutput(global_step=1377, training_loss=0.45390338108170786, metrics={'train_runtime': 272.3305, 'train_samples_per_second': 40.407, 'train_steps_per_second': 5.056, 'total_flos': 405114969714960.0, 'train_loss': 0.45390338108170786, 'epoch': 3.0})